# StrataForge Progress Notebook
## major-changes-v2 Phase F-J
Purpose: exercise the acquisition-only tree runtime, semantic summary artifacts, and observability event capture after the v1 parse/tree cutover.

### Environment Assumptions
- This notebook is deterministic and does not require network access.
- It uses the committed born-digital Phase 01 PDF fixture, so no OCR runtime is required.
- It uses the acquisition-only tree build path plus a noop summary gateway and local JSONL event logging.

In [ ]:
# environment setup
from pathlib import Path

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "major-changes-v2-phase-fj"
ACQUISITION_ROOT = ARTIFACT_ROOT / "acquisition_runs"
for path in (ARTIFACT_ROOT, ACQUISITION_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print(
    {
        "repo_root": str(REPO_ROOT),
        "artifact_root": str(ARTIFACT_ROOT),
        "acquisition_root": str(ACQUISITION_ROOT),
    }
)

In [ ]:
# imports
import json

from strataforge.domain import AcquisitionRequest, AcquisitionSettings, TreeBuildRequest
from strataforge.ingest import acquire_document
from strataforge.ingest.acquisition_artifacts import settings_digest
from strataforge.ingest.fingerprint import fingerprint_document
from strataforge.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayService,
    LiteLLMProviderConfig,
    NoopProviderAdapter,
    NoopScriptedResponse,
)
from strataforge.observability import EventBus, JsonLoggerSubscriber
from strataforge.tree import build_tree

In [ ]:
# configuration
PDF_PATH = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"
PDF_FINGERPRINT = fingerprint_document(str(PDF_PATH))
SETTINGS_DIGEST = settings_digest(AcquisitionSettings())
RUN_SUFFIX = f"{PDF_FINGERPRINT.sha256[:8]}-{SETTINGS_DIGEST[:8]}"
ACQUISITION_RUN_ID = f"progress-major-changes-v2-acquisition-{RUN_SUFFIX}"
TREE_RUN_ID = f"progress-major-changes-v2-tree-{RUN_SUFFIX}"

In [ ]:
# execution
event_log_path = ARTIFACT_ROOT / "events.jsonl"
event_bus = EventBus(subscribers=(JsonLoggerSubscriber(str(event_log_path)),))
summary_gateway = GatewayService(
    GatewayConfig(
        provider=LiteLLMProviderConfig(model="progress-noop"),
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "summarize_leaf_node": NoopScriptedResponse(
                output_json={"summary": "leaf summary", "keywords": ["leaf"]}
            ),
            "summarize_parent_node": NoopScriptedResponse(
                output_json={"summary": "parent summary", "keywords": ["parent"]}
            ),
        }
    ),
)
acquisition_manifest = acquire_document(
    AcquisitionRequest(
        source_path=str(PDF_PATH),
        acquisition_run_id=ACQUISITION_RUN_ID,
        artifact_root=str(ACQUISITION_ROOT),
    ),
    event_bus=event_bus,
)
tree_manifest = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=str(Path(acquisition_manifest.artifact_root) / "manifest.json"),
        tree_run_id=TREE_RUN_ID,
        summarize=True,
    ),
    gateway=summary_gateway,
    event_bus=event_bus,
)
acquisition_projection = json.loads(
    Path(acquisition_manifest.projection_view_path or "").read_text(encoding="utf-8")
)
node_cards = json.loads(Path(tree_manifest.node_cards_path).read_text(encoding="utf-8"))
node_summaries = json.loads(
    Path(tree_manifest.node_summaries_path or "").read_text(encoding="utf-8")
)
event_lines = [
    json.loads(line)
    for line in event_log_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

In [ ]:
# inspect results
summary = {
    "acquisition": {
        "document_id": acquisition_manifest.document_id,
        "page_count": acquisition_manifest.page_count,
        "selected_outline_source": acquisition_manifest.selected_outline_source.value,
        "projection_page_count": len(acquisition_projection["pages"]),
        "projection_line_counts": [len(page["lines"]) for page in acquisition_projection["pages"]],
    },
    "tree": {
        "titles": [card["title"] for card in node_cards],
        "summary_methods": [card["summary_method"] for card in node_cards],
        "tokenizer_identities": [item["tokenizer_identity"] for item in node_summaries],
        "summary_artifact": tree_manifest.node_summaries_path,
    },
    "events": {
        "count": len(event_lines),
        "names": sorted({event["event_name"] for event in event_lines}),
    },
}
print(json.dumps(summary, indent=2, sort_keys=True))

### Known Limitations
- This notebook exercises the acquisition-only path on a born-digital fixture only.
- Semantic summaries still use the heuristic tokenizer by default unless an optional exact tokenizer is installed and selected.
- Optional edge exporters and multimodal enrichment are not exercised here because they depend on extra adapters or optional third-party packages.